# EDA 자동화 파이프라인

**PydanticAI + Google Gemini 기반 범용 탐색적 데이터 분석(EDA) 자동화 도구**

---

## 사용 방법

1. **`1. 사용자 설정` 셀만 수정**하여 분석할 파일 경로와 이름을 입력합니다.
2. **상단 메뉴 → Run All** 로 전체 실행합니다.
3. 분석 결과는 화면에 출력되고 `data/` 폴더에 JSON으로 자동 저장됩니다.

## 지원 파일 형식

| 형식 | 확장자 | 비고 |
|------|--------|---------|
| CSV | `.csv` | 인코딩 자동 감지 |
| JSON | `.json` | 1차원 표 구조 |
| Excel | `.xlsx`, `.xls` | 첫 번째 시트 |

## 전제 조건

- `.env` 파일에 `GEMINI_API_KEY` 설정 필요
- 데이터는 **행·열 표 구조**여야 합니다 (중첩 JSON 불가)

---
## 0. 환경 준비

In [1]:
import os
import json
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

load_dotenv(override=True)
gemini_model = os.getenv('GEMINI_MODEL', 'gemini-3.1-flash-lite-preview')
model_id = f'google-gla:{gemini_model}'

if not os.getenv('GEMINI_API_KEY'):
    raise ValueError('.env 파일에 GEMINI_API_KEY가 설정되어 있지 않습니다.')

print(f'✅ 환경 준비 완료')
print(f'   사용 모델: {model_id}')

✅ 환경 준비 완료
   사용 모델: google-gla:gemini-3.1-flash-lite-preview


---
## 1. 사용자 설정

> **여기만 수정하면 됩니다.** 나머지 셀은 수정하지 않아도 됩니다.

In [2]:
# =====================================================================
# 필수 설정
# =====================================================================
FILE_PATH   = 'data/prices.csv'       # 분석할 파일 경로 (.csv / .json / .xlsx)
DATASET_NAME = '금융 자산 가격 데이터'  # 리포트에 표시될 데이터셋 이름

# =====================================================================
# 선택 설정
# =====================================================================
DATE_COLUMN  = 'Date'   # 날짜 컬럼명. 없으면 None 으로 설정
SHEET_NAME   = 0        # Excel 시트 번호 또는 이름 (Excel 파일일 때만 사용)
SAVE_REPORT  = True     # True: JSON 리포트 자동 저장
SAVE_DIR     = 'data'   # 리포트 저장 폴더

---
## 2. 데이터 로드

In [4]:
def load_data(file_path: str, date_column: str | None = None, sheet_name=0) -> pd.DataFrame:
    """파일 확장자에 따라 자동으로 데이터를 로드합니다."""
    ext = os.path.splitext(file_path)[1].lower()

    if not os.path.exists(file_path):
        raise FileNotFoundError(f'파일을 찾을 수 없습니다: {file_path}')

    if ext == '.csv':
        try:
            df = pd.read_csv(file_path, encoding='utf-8')
        except UnicodeDecodeError:
            df = pd.read_csv(file_path, encoding='cp949')
    elif ext == '.json':
        df = pd.read_json(file_path)
    elif ext in ('.xlsx', '.xls'):
        df = pd.read_excel(file_path, sheet_name=sheet_name)
    else:
        raise ValueError(f'지원하지 않는 파일 형식입니다: {ext}')

    if date_column and date_column in df.columns:
        df[date_column] = pd.to_datetime(df[date_column], errors='coerce')
        df = df.set_index(date_column)

    return df


df = load_data(FILE_PATH, DATE_COLUMN, SHEET_NAME)

print(f'✅ 데이터 로드 완료: {FILE_PATH}')
print(f'   크기: {df.shape[0]:,}행 × {df.shape[1]}열')
print(f'   컬럼: {df.columns.tolist()}')
if isinstance(df.index, pd.DatetimeIndex):
    print(f'   기간: {df.index.min().date()} ~ {df.index.max().date()}')
df.head(5)

✅ 데이터 로드 완료: data/prices.csv
   크기: 1,825행 × 13열
   컬럼: ['SPY', 'QQQ', 'TLT', 'AGG', 'GLD', 'EEM', 'CL=F', 'GC=F', 'SI=F', 'BTC-USD', 'ETH-USD', '^VIX', 'DX-Y.NYB']
   기간: 2021-01-01 ~ 2025-12-30


,SPY,QQQ,TLT,AGG,GLD,EEM,CL=F,GC=F,SI=F,BTC-USD,ETH-USD,^VIX,DX-Y.NYB
Date,,,,,,,,,,,,,
2021-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29374.152344,730.367554,NaN,NaN
2021-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32127.267578,774.534973,NaN,NaN
2021-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32782.023438,975.507690,NaN,NaN
2021-01-04,343.319183,300.163055,133.755646,101.107834,182.330002,46.271252,47.619999,1944.699951,27.284000,31971.914062,1040.233032,26.969999,89.879997
2021-01-05,345.683685,302.637726,132.762253,101.005074,182.869995,47.383537,49.930000,1952.699951,27.570999,33992.429688,1100.006104,25.340000,89.440002


---
## 3. EDA 리포트 스키마

Pydantic `BaseModel` 기반 구조화 출력 스키마입니다.  
에이전트는 이 스키마에 맞춰 분석 결과를 자동으로 채워 반환합니다.

In [5]:
class DataOverview(BaseModel):
    rows: int                        = Field(description='전체 행 수')
    columns: int                     = Field(description='전체 열 수')
    numeric_columns: list[str]       = Field(description='수치형 컬럼 목록')
    categorical_columns: list[str]   = Field(description='범주형 컬럼 목록')
    total_missing_cells: int         = Field(description='전체 결측 셀 수')


class MissingInfo(BaseModel):
    column_name: str    = Field(description='컬럼명')
    missing_count: int  = Field(description='결측치 개수')
    missing_ratio: float = Field(description='결측 비율 (0.0~1.0)')
    recommendation: str = Field(description='처리 권장사항')


class OutlierInfo(BaseModel):
    column_name: str     = Field(description='컬럼명')
    outlier_count: int   = Field(description='이상치 개수')
    outlier_ratio: float = Field(description='이상치 비율 (0.0~1.0)')
    lower_bound: float   = Field(description='IQR 하한 경계')
    upper_bound: float   = Field(description='IQR 상한 경계')
    recommendation: str  = Field(description='처리 권장사항')


class CorrelationHighlight(BaseModel):
    col_a: str          = Field(description='첫 번째 컬럼명')
    col_b: str          = Field(description='두 번째 컬럼명')
    correlation: float  = Field(description='상관계수')
    interpretation: str = Field(description='상관관계 해석')


class KeyInsight(BaseModel):
    category: str  = Field(description='인사이트 분류 (결측치/이상치/분포/상관관계 등)')
    insight: str   = Field(description='발견한 인사이트 내용')
    priority: str  = Field(description='중요도: High/Medium/Low')


class EDAReport(BaseModel):
    dataset_name: str                            = Field(description='데이터셋 이름')
    overview: DataOverview
    missing_analysis: list[MissingInfo]          = Field(description='결측치 분석 (결측 없으면 빈 리스트)')
    outlier_analysis: list[OutlierInfo]          = Field(description='이상치 분석 결과')
    correlation_highlights: list[CorrelationHighlight] = Field(description='주목할 상관관계 상위 3~5개')
    key_insights: list[KeyInsight]               = Field(description='핵심 인사이트 3~5개')
    next_steps: list[str]                        = Field(description='권장 후속 작업 목록')

print('✅ 스키마 정의 완료')

✅ 스키마 정의 완료


---
## 4. Tool 함수 정의

에이전트가 호출할 분석 함수 5종입니다. 모든 함수는 `pd.DataFrame`을 입력받아  
어떤 데이터에도 동작하도록 설계되어 있습니다.

In [6]:
def describe_data(df: pd.DataFrame) -> dict:
    """데이터 형태, 수치형 요약 통계, 범주형 요약을 반환합니다."""
    num_cols = df.select_dtypes(include='number').columns.tolist()
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    return {
        'shape': {'rows': int(df.shape[0]), 'columns': int(df.shape[1])},
        'numeric_columns': num_cols,
        'categorical_columns': cat_cols,
        'numeric_summary': df[num_cols].describe().round(2).to_dict() if num_cols else {},
        'categorical_summary': {
            col: {
                'unique_count': int(df[col].nunique()),
                'top_value': str(df[col].mode()[0]) if not df[col].mode().empty else None
            } for col in cat_cols
        }
    }


def check_missing(df: pd.DataFrame) -> dict:
    """각 컬럼의 결측치 개수와 비율(0~1)을 반환합니다."""
    mc = df.isnull().sum()
    mr = (mc / len(df)).round(4)
    missing_df = pd.DataFrame({'count': mc, 'ratio': mr}).query('count > 0').sort_values('count', ascending=False)
    return {
        'total_missing_cells': int(df.isnull().sum().sum()),
        'columns_with_missing': missing_df.to_dict(orient='index')
    }


def detect_outliers_all(df: pd.DataFrame) -> dict:
    """모든 수치형 컬럼에 IQR 기준 이상치 탐지를 수행합니다."""
    results = {}
    for col in df.select_dtypes(include='number').columns:
        s = df[col].dropna()
        if len(s) < 4:
            continue
        Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
        IQR = Q3 - Q1
        lo, hi = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        out = s[(s < lo) | (s > hi)]
        results[col] = {
            'outlier_count': len(out),
            'outlier_ratio': round(len(out) / len(s), 4),
            'lower_bound': round(float(lo), 3),
            'upper_bound': round(float(hi), 3)
        }
    return results


def correlation_top(df: pd.DataFrame, method: str = 'pearson', top_n: int = 5) -> list:
    """수치형 컬럼 간 상관관계 상위 N쌍을 반환합니다."""
    num = df.select_dtypes(include='number')
    if num.shape[1] < 2:
        return []
    corr = num.corr(method=method).round(3)
    pairs = [
        {'col_a': corr.columns[i], 'col_b': corr.columns[j], 'corr': float(corr.iloc[i, j])}
        for i in range(len(corr.columns))
        for j in range(i + 1, len(corr.columns))
    ]
    pairs.sort(key=lambda x: abs(x['corr']), reverse=True)
    return pairs[:top_n]


def visualize_summary(df: pd.DataFrame) -> dict:
    """수치형 컬럼 분포를 한 번에 시각화합니다 (최대 6개)."""
    num_cols = df.select_dtypes(include='number').columns.tolist()
    n = min(len(num_cols), 6)
    if n == 0:
        return {'message': '수치형 컬럼이 없습니다.'}

    ncols = 3
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
    axes = np.array(axes).flatten()

    for i, col in enumerate(num_cols[:n]):
        s = df[col].dropna()
        s.plot(kind='hist', bins=30, ax=axes[i], color='steelblue', alpha=0.7)
        axes[i].axvline(s.mean(), color='red', linestyle='--', linewidth=1.2, label=f'평균 {s.mean():.2f}')
        axes[i].set_title(col, fontsize=10)
        axes[i].legend(fontsize=8)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('수치형 컬럼 분포 요약', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()
    return {'visualized_columns': num_cols[:n]}

print('✅ Tool 함수 5종 정의 완료')

✅ Tool 함수 5종 정의 완료


---
## 5. 에이전트 구성

In [7]:
@dataclass
class DataDeps:
    df: pd.DataFrame
    dataset_name: str


def build_eda_agent() -> Agent:
    """EDA 자동화 에이전트를 생성하고 Tool 5종을 등록하여 반환합니다."""
    agent = Agent(
        model=model_id,
        deps_type=DataDeps,
        output_type=EDAReport,
        system_prompt=(
            'EDA 자동화 에이전트입니다. '
            '반드시 모든 Tool을 실행한 뒤 실제 수치를 기반으로 EDAReport를 채우십시오. '
            '추측이나 가정 없이 Tool 실행 결과만 사용하십시오. '
            '비율 값은 0.0~1.0 사이의 소수로 입력하십시오.'
        )
    )

    @agent.tool
    def tool_describe_data(ctx: RunContext[DataDeps]) -> dict:
        """데이터 형태, 컬럼 목록, 수치형·범주형 요약 통계를 반환합니다."""
        return describe_data(ctx.deps.df)

    @agent.tool
    def tool_check_missing(ctx: RunContext[DataDeps]) -> dict:
        """각 컬럼의 결측치 개수와 비율(0~1)을 분석합니다."""
        return check_missing(ctx.deps.df)

    @agent.tool
    def tool_detect_outliers_all(ctx: RunContext[DataDeps]) -> dict:
        """모든 수치형 컬럼에서 IQR 기준 이상치를 일괄 탐지합니다."""
        return detect_outliers_all(ctx.deps.df)

    @agent.tool
    def tool_correlation_top(ctx: RunContext[DataDeps], method: str = 'pearson') -> list:
        """수치형 컬럼 간 상관관계 상위 5쌍을 반환합니다."""
        return correlation_top(ctx.deps.df, method)

    @agent.tool
    def tool_visualize_summary(ctx: RunContext[DataDeps]) -> dict:
        """수치형 컬럼 분포를 한 번에 시각화합니다."""
        return visualize_summary(ctx.deps.df)

    return agent

print('✅ 에이전트 빌더 준비 완료')

✅ 에이전트 빌더 준비 완료


---
## 6. 파이프라인 실행

In [8]:
async def run_eda_pipeline(df: pd.DataFrame, dataset_name: str) -> tuple:
    """데이터프레임을 받아 EDA 리포트와 소요 시간을 반환합니다."""
    deps = DataDeps(df=df, dataset_name=dataset_name)
    agent = build_eda_agent()

    print(f'🤖 EDA 에이전트 실행 중... ({dataset_name})')
    start = time.time()
    result = await agent.run(
        f"'{dataset_name}' 데이터에 대해 완전한 EDA 리포트를 생성해주세요. "
        f'모든 Tool을 순서대로 실행하고 실제 수치를 사용해 EDAReport를 완성하십시오.',
        deps=deps
    )
    elapsed = time.time() - start
    print(f'✅ 완료 (소요: {elapsed:.1f}초)')
    return result.output, elapsed


report, elapsed = await run_eda_pipeline(df, DATASET_NAME)

🤖 EDA 에이전트 실행 중... (금융 자산 가격 데이터)
✅ 완료 (소요: 12.4초)


---
## 7. 리포트 출력

In [9]:
def print_report(report: EDAReport, elapsed: float):
    sep = '=' * 65
    print(sep)
    print(f'  📋 EDA 리포트: {report.dataset_name}  [{elapsed:.1f}초]')
    print(sep)

    o = report.overview
    print(f'\n▶ 데이터 개요')
    print(f'  크기       : {o.rows:,}행 × {o.columns}열')
    print(f'  수치형 컬럼: {o.numeric_columns}')
    print(f'  범주형 컬럼: {o.categorical_columns}')
    print(f'  총 결측 셀 : {o.total_missing_cells:,}개')

    print(f'\n▶ 결측치 분석')
    if report.missing_analysis:
        for m in report.missing_analysis:
            print(f'  • {m.column_name}: {m.missing_count:,}개 ({m.missing_ratio:.1%}) → {m.recommendation}')
    else:
        print('  결측치 없음')

    print(f'\n▶ 이상치 분석')
    for o in report.outlier_analysis:
        if o.outlier_count > 0:
            print(f'  • {o.column_name}: {o.outlier_count:,}개 ({o.outlier_ratio:.1%})')
            print(f'    경계: [{o.lower_bound} ~ {o.upper_bound}]')
            print(f'    → {o.recommendation}')

    print(f'\n▶ 주요 상관관계')
    for c in report.correlation_highlights:
        print(f'  • {c.col_a} ↔ {c.col_b}: {c.correlation:.3f}  ({c.interpretation})')

    print(f'\n▶ 핵심 인사이트')
    for i, ins in enumerate(report.key_insights, 1):
        print(f'  {i}. [{ins.priority}] [{ins.category}] {ins.insight}')

    print(f'\n▶ 권장 후속 작업')
    for i, s in enumerate(report.next_steps, 1):
        print(f'  {i}. {s}')
    print()


print_report(report, elapsed)

  📋 EDA 리포트: 금융 자산 가격 데이터  [12.4초]

▶ 데이터 개요
  크기       : 1,825행 × 13열
  수치형 컬럼: ['SPY', 'QQQ', 'TLT', 'AGG', 'GLD', 'EEM', 'CL=F', 'GC=F', 'SI=F', 'BTC-USD', 'ETH-USD', '^VIX', 'DX-Y.NYB']
  범주형 컬럼: []
  총 결측 셀 : 6,273개

▶ 결측치 분석
  • SPY, QQQ, GLD, EEM, TLT, AGG, ^VIX: 571개 (31.3%) → 해당 자산들은 데이터 수집 시작 시점이 다른 것으로 보이므로, 분석 목적에 따라 기간을 필터링하거나 시계열 보간법 적용 필요.
  • CL=F, GC=F, SI=F, DX-Y.NYB: 569개 (31.2%) → 상기 자산들과 마찬가지로 데이터 정렬 및 결측치 처리(필요 시 제거 혹은 보간)가 선행되어야 함.

▶ 이상치 분석
  • SI=F: 74개 (5.9%)
    경계: [11.45 ~ 42.02]
    → 은 가격은 변동성이 크므로 단순 제거보다는 로그 변환이나 윈저라이징 고려.
  • CL=F: 68개 (5.4%)
    경계: [46.42 ~ 103.22]
    → 원유 가격 이상치 확인 및 거시경제 이벤트 영향 검토.
  • ^VIX: 38개 (3.0%)
    경계: [6.19 ~ 31.42]
    → 공포 지수인 ^VIX는 시장 충격 시 급등하므로, 이상치 제거 시 주요 경제 위기 데이터가 손실될 수 있으므로 주의.

▶ 주요 상관관계
  • GLD ↔ GC=F: 1.000  (금 ETF(GLD)와 금 선물(GC=F)은 완벽한 양의 상관관계를 보임.)
  • SPY ↔ QQQ: 0.991  (주식 시장을 대표하는 S&P 500(SPY)과 나스닥 100(QQQ)은 매우 강한 동조화 현상을 보임.)
  • GLD ↔ SI=F: 0.931  (금과 은(SI=F)은 귀금속으로서 강한 상관관계를 가짐.)
  • GC=F ↔ SI=F: 0.930 

---
## 8. 결과 저장

In [10]:
if SAVE_REPORT:
    os.makedirs(SAVE_DIR, exist_ok=True)
    base_name = os.path.splitext(os.path.basename(FILE_PATH))[0]
    save_path = os.path.join(SAVE_DIR, f'eda_report_{base_name}.json')

    with open(save_path, 'w', encoding='utf-8') as f:
        json.dump(report.model_dump(), f, ensure_ascii=False, indent=2)

    print(f'✅ 리포트 저장 완료: {save_path}')
else:
    print('SAVE_REPORT = False 설정으로 저장을 건너뜁니다.')

✅ 리포트 저장 완료: data\eda_report_prices.json


---
## 9. 정확도 검증

에이전트 출력값을 실제 계산값과 비교합니다.

In [11]:
actual_rows     = len(df)
actual_cols     = df.shape[1]
actual_missing  = int(df.isnull().sum().sum())
actual_num_cols = df.select_dtypes(include='number').shape[1]

checks = [
    ('행 수',       actual_rows,     report.overview.rows),
    ('열 수',       actual_cols,     report.overview.columns),
    ('총 결측 셀',  actual_missing,  report.overview.total_missing_cells),
    ('수치형 컬럼 수', actual_num_cols, len(report.overview.numeric_columns)),
]

print(f"{'항목':<20} {'실측값':>10} {'에이전트':>10} {'일치':>6}")
print('-' * 50)
for label, actual, pred in checks:
    match = '✅' if actual == pred else '❌'
    print(f'{label:<20} {actual:>10,} {pred:>10,} {match:>6}')

항목                          실측값       에이전트     일치
--------------------------------------------------
행 수                       1,825      1,825      ✅
열 수                          13         13      ✅
총 결측 셀                    6,273      6,273      ✅
수치형 컬럼 수                     13         13      ✅
